# EDA Ventas Cruzadas (con Desanonimización autorizada)

Este notebook se conecta a la vista `ml.v_ventas_cruzadas_desanonima` para poder visualizar temporalmente los nombres reales y ayudar en el EDA, pero genera el pipeline final basándose estrictamente en el `hash_anonimo`.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
import joblib
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Extracción desde la vista segura (Via Docker Compose Network)
pg_user = os.getenv('PG_USER', 'etl_user')
pg_pass = os.getenv('PG_PASSWORD', 'password')
pg_host = os.getenv('PG_HOST', 'postgres_edw')
pg_port = os.getenv('PG_PORT', '5432')
pg_db = os.getenv('PG_DB', 'edw')

engine = create_engine(f'postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}')
query = "SELECT hash_anonimo, nombre_cliente, nombre_articulo, cantidad FROM ml.v_ventas_cruzadas_desanonima"
df = pd.read_sql(query, engine)

print("Datos Desanonimizados para EDA:")
display(df.head())

In [ ]:
# 2. Preparación para Apriori
# Agrupamos por hash_anonimo (para mantener compatibilidad en producción)
basket = (df.groupby(['hash_anonimo', 'nombre_articulo'])['cantidad']
          .sum().unstack().reset_index().fillna(0)
          .set_index('hash_anonimo'))

def encode_units(x):
    return 1 if x >= 1 else 0

basket_sets = basket.applymap(encode_units)

In [ ]:
# 3. Entrenamiento (Apriori)
frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

display(rules.sort_values('lift', ascending=False).head(10))

In [ ]:
# 4. Serializar modelo basado EN REGLAS (sin PII)
rules_export = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
joblib.dump(rules_export, '../models/reglas_ventas_cruzadas.joblib')
print("Modelo serializado como reglas_ventas_cruzadas.joblib (100% anonymizado)")